# bridge_write

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): the report and the figure of `bridge_report`'s summaries, and synthetic rows with known effects. `writeC1` (`report.md`: Gate C1 with its robustness, the inert-bridge flag, H20-3 to H20-8, the per-arm and core measures, part b, and every `C1_PLAN` item), `figC1` (`c1.png`: honest pooled share per arm with the lead and the half-lead bar; the clean boards beside), `syntheticRows20` (every planned C1 cell) and `syntheticPartB` (part b's rows), for `test_c1_report` and the quick demo.

In [ ]:
#| default_exp bridge_write

In [ ]:
#| hide
from nbdev.showdoc import *

## The report

In [ ]:
#| export
import json

import numpy as np

from explRL.bridge_grid import ARMS20, PARTB_MODELS
from explRL.bridge_plan import C1_PLAN
from explRL.bridge_report import N20, PARTB_TESTS, planned20
from explRL.colony_report import NAN
from explRL.rl17_grid import BIG_SEEDS

_g = lambda x: 'n/a' if x is None or not isinstance(x, (int, float)) or not np.isfinite(x) else f'{x:.3f}'
_ci = lambda c: f'[{_g(c[0])}, {_g(c[1])}]'
_row = lambda name, t, v='': f"| {name} | {t['n']} | {_g(t['estimate'])} | {_ci(t['ci'])} | {_g(t['p'])} | {v} |"
_HEAD = ['| item | boards | estimate | 95% interval | p (one-sided, exact, uncorrected) | verdict |', '|---|---|---|---|---|---|']


def _partB(B):
    h = B['H20_6']
    L = ['## Part b (H20-6, descriptive): 3B at 32 agents, squads of 4 against one board', '',
         f"Gain over the warm start minus on one board: {_g(h['estimate'])} over {h['n']} runs ({h['positive']} positive), "
         f"sign-flip p {_g(h['p'])}; mean gain in squads of 4 {_g(h['gain_squads4'])}, on one board {_g(h['gain_oneBoard'])}; "
         f"Phase 17's gain at 4 agents {_g(B['phase17'][4])}, at 32 {_g(B['phase17'][32])}. Honest loss with liars: squads of 4 "
         f"{_g(h['honestLoss_squads4'])}, one board {_g(h['honestLoss_oneBoard'])}.", '',
         '| model | welfare, squads of 4 | gain | honest loss | welfare, one board | gain | honest loss | difference |', '|' + '---|' * 8]
    L += [f"| {m} | " + ' | '.join(_g(v[a][k]) for a in ('squads4', 'oneBoard') for k in ('welfare', 'gain', 'honestLoss')) +
          f" | {_g(v['diff'])} |" for m, v in B['runs'].items()]
    return L + ['']


def writeC1(path, S, meta, plan=C1_PLAN, partB=None):
    """report.md (FR-D4): Gate C1 with its robustness, the verdict and the inert-bridge flag, H20-3 to H20-5, H20-7 and H20-8,
    each arm's honest pooled share, advantage, columns and core measures, part b when given (S None: part b alone), and every
    plan item."""
    L = ['# Phase 20 (C1): squads with coupled work and bridges (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}, boards {S['boards'] if S else 0}", '']
    if S:
        G, H = S['gate'], S['hyp']
        L += ['## Gate C1 (64 agents, independent liars, seeds 1800-1807)', ''] + _HEAD
        for m, g in ((G['metric'], G), ('welfareHonest (robustness)', G['welfareHonest'])):
            L += [_row(f"(i) bridge1 - colonyGuards, {m}", g['i'], f"passes: {g['i']['passes']}"),
                  _row(f"(ii) margin: (bridge1 - colonyGuards) - 0.5 (squads - colonyGuards), {m}", g['ii'],
                       f"holds: {g['ii']['holds']} (bridge1 {_g(g['ii']['bridge'])} against the bar {_g(g['ii']['bar'])})"),
                  _row(f'lead: squads - colonyGuards, {m}', g['ii']['lead'], f"plan's lead {plan['gate']['lead']}" if g is G else '')]
        L += ['', f"**Gate C1: {'passes' if G['passes'] else 'fails'}** (robust on raw honest welfare: part i {G['robust']['i']}, "
              f"part ii {G['robust']['ii']}). **Inert-bridge flag: {'fires: stop and ask before A1 adopts bridges' if S['inert']['fires'] else 'does not fire'}** "
              f"(bridge1 - coupled {_g(S['inert']['estimate'])} on seeds 1800-1804).", '',
              '## Secondary (seeds 1800-1804; exploratory, no Holm)', ''] + _HEAD
        h3, h4, h5, h7 = H['H20_3'], H['H20_4'], H['H20_5'], H['H20_7']
        L += [_row('H20-3 margin: (bridge1 - coupled) - 0.5 (squads - coupled)', h3,
                   f"win-back {_g(h3['winBack'])} {_ci(h3['winBackCi'])}; supported: {h3['supported']}"),
              _row('coupling cost: squads - coupled', h3['couplingCost']), _row('bridge gain: bridge1 - coupled', h3['bridgeGain']),
              _row('H20-4 margin: (bridge2 - bridge1) - 0.25 (squads - coupled), p for < 0', h4, f"supported: {h4['supported']}"),
              _row('second bridge: bridge2 - bridge1', h4['secondBridge']),
              _row('H20-5 false-relay share: bridge1 - bridgeGuard', h5, f"supported: {h5['supported']}"),
              _row('H20-5 honestPooled: bridgeGuard - bridge1', h5['pooled']),
              _row('H20-7 cross-squad contamination: bridge1 - bridgeGuard', h7['guarding'])]
        L += ['', 'H20-7, cross-squad contamination per arm: ' + '; '.join(f'{a} {_g(v[0])} {_ci(v[1:])}' for a, v in h7['perArm'].items()), '',
              '## H20-8: cost without liars (clean boards, seeds 1800-1804; descriptive, not in Gate C1)', '',
              'Per arm: ' + '; '.join(f"{a} honestPooled {_g(v['honestPooled'])}, welfareHonest {_g(v['welfareHonest'])}" for a, v in H['H20_8']['perArm'].items()),
              ''] + _HEAD + [_row(f'{c}, {m} (p for < 0: a cost)', t) for c, d in H['H20_8']['contrasts'].items() for m, t in d.items()]
        A = S['arms']
        L += ['', '## Per arm (independent liars)', '', '| arm | boards | honestPooled | 95% interval | advantage over colonyGuards | 95% interval |',
              '|---|---|---|---|---|---|']
        L += [f"| {a} | {v['boards']} | {_g(v['honestPooled'][0])} | {_ci(v['honestPooled'][1:])} | "
              + (f"{_g(v['advantage']['estimate'])} | {_ci(v['advantage']['ci'])} |" if v['advantage'] else '| |') for a, v in A.items()]
        for title, k in (('Core measures (FR-D5), boards with liars', 'core'), ('Core measures, clean boards', 'coreClean'),
                         ('Phase 18 and bridge columns (means over boards with liars)', 'columns')):
            ks = list(next(v[k] for v in A.values() if v[k]))
            L += ['', f'## {title}', '', '| arm | ' + ' | '.join(ks) + ' |', '|---|' + '---|' * len(ks)]
            L += [f'| {a} | ' + ' | '.join(_g(v[k][c]) for c in ks) + ' |' for a, v in A.items() if v[k]]
        L += ['', 'Note (FR-D5): a mirror post is not a board share, so the per-agent shares, sharing per held clue and Gorelick\'s index '
              'count it as an idle turn; this biases collaboration and division of labour slightly against the coupled arms. '
              'Mirror posts are in `mirrorPosts` and `mirrorPostsPerHolder`.']
        L += ['']
    L += _partB(partB) if partB else []
    L += ['## C1_PLAN', ''] + [f'- {k}: {json.dumps(v, default=str)}' for k, v in plan.items()]
    open(path, 'w').write('\n'.join(L) + '\n')

## The figure

In [ ]:
#| export
def figC1(out, S):
    "c1.png (FR-D4): honest pooled share per arm with liars (mean, 95% interval, boards), with the lead and the half-lead bar; clean boards beside."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    A, G = S['arms'], S['gate']
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw=dict(width_ratios=[2, 1]))
    ref = A['colonyGuards']['honestPooled'][0]
    for x, (arm, v) in enumerate(A.items()):
        e, lo, hi = v['honestPooled']
        a.errorbar([x], [e], yerr=[[e - lo], [hi - e]], fmt='o', color='#1f6391' if 'bridge' in arm else '#555555', capsize=3, ms=8)
    a.axhline(ref, color='#555555', lw=0.8, ls=':', label='colonyGuards')
    a.axhline(ref + G['ii']['lead']['estimate'], color='#2e8b57', lw=1, ls='--', label='squads: the lead')
    a.axhline(ref + G['ii']['bar'], color='#c0392b', lw=1, ls='--', label='half the lead: the bar of part (ii)')
    a.set(xticks=range(len(A)), ylabel='honest pooled share', title=f"64 agents with liars; Gate C1 {'passes' if G['passes'] else 'fails'}")
    a.set_xticks(range(len(A)), list(A), rotation=30, ha='right', fontsize=8)
    a.legend(fontsize=7, frameon=False)
    cl = S['hyp']['H20_8']['perArm']
    b.bar(range(len(cl)), [v['honestPooled'] for v in cl.values()], 0.5, color='#7f7f7f')
    b.set_xticks(range(len(cl)), list(cl), fontsize=8)
    b.set(ylabel='honest pooled share', title='Clean boards (H20-8)')
    for ax in (a, b):
        ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{out}/c1.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

## Synthetic rows

In [ ]:
#| export
def _slots(nLiar, N=N20):
    "Per-slot JSON columns (contestColumns' layout): honest even slots share more than odd ones, guards contest."
    j = lambda f: json.dumps([0 if i < nLiar else f(i) for i in range(N)])
    return dict(agentTurns=json.dumps([40] * N), agentShares=j(lambda i: 20 if i % 2 == 0 else 5),
                agentContests=j(lambda i: 2 if nLiar <= i < nLiar + 16 else 0), agentSubmits=j(lambda i: 5), agentOpens=j(lambda i: 5))


def syntheticRows20(lead=0.171, cost=0.10, back=0.07, second=0.01, guard=0.005, clean=(0.03, 0.01), falseRelay=(0.3, 0.1),
                    contamination=(0.08, 0.03), noise=0.02, seed=0, plan=C1_PLAN):
    """Rows of every planned C1 cell (N 64; planned20) with known effects in honestPooled (welfareHonest 1.3 times it, welfare
    equal): colonyGuards 0.07; squads = colonyGuards + lead; coupled = squads - cost; bridge1 = coupled + back; bridge2 =
    bridge1 + second; bridgeGuard = bridge1 + guard. Clean boards: squads 0.40, coupled 0.40 - clean[0], bridge1 0.40 - clean[1].
    False-relay share and cross-squad contamination: bridge1 and bridge2 at [0], bridgeGuard at [1], coupled no relays (NaN)
    and no contamination (0). One noise draw per seed is shared by every arm, so paired differences are exact. The core
    measures' columns are the same on every board (gullibility 6 / honestTurns, suppression 2 / 4, and so on)."""
    rng, rows = np.random.default_rng(seed), []
    e = {s: rng.normal(0, noise) for s in sorted({s for _, s in planned20(plan)})}
    hp = dict(colonyGuards=0.07, squads=0.07 + lead)
    hp.update(coupled=hp['squads'] - cost, bridge1=hp['squads'] - cost + back)
    hp.update(bridge2=hp['bridge1'] + second, bridgeGuard=hp['bridge1'] + guard)
    hc = dict(squads=0.40, coupled=0.40 - clean[0], bridge1=0.40 - clean[1])
    pick = lambda a, v: v[1] if a == 'bridgeGuard' else 0.0 if a == 'coupled' else v[0]
    for lv, s in sorted(planned20(plan)):
        a, f = lv.split('|')
        nl, h = (0, hc[a]) if f == 'clean' else (16, hp[a])
        x, cp = h + e[s], ARMS20[a].get('couple')
        rows.append(dict(level=lv, seed=s, N=N20, nLiar=nl, tEnd=40.0, honestPooled=x, welfareHonest=1.3 * x, welfare=x,
                         falseRelayShare=pick(a, falseRelay) if ARMS20[a].get('bridges') else NAN,
                         crossContamination=pick(a, contamination) if cp else NAN, bridgeRelays=40.0 if ARMS20[a].get('bridges') else 0.0,
                         mirrorPosts=200.0 if cp else 0.0, contamination=0.3, inTokPerCall=780.0, honestTurns=40.0 * (N20 - nl),
                         misinformedFirst=6.0 if nl else 0.0, liarFirst=4.0 if nl else 0.0, uncorrected=2.0 if nl else 0.0,
                         liarPosts=20.0 if nl else 0.0, contests=10.0 if nl else 0.0, guardLies=8.0 if nl else 0.0,
                         contestLags=json.dumps([1.0, 3.0] if nl else []), wrongHonest=6.0, sameWrong=2.0, sharePerHeld=0.5,
                         inputTokens=1.5e6, outputTokens=3e5, solved=30.0, **_slots(nl)))
    return rows


def syntheticPartB(gain4=0.05, gain1=0.006, loss=(0.04, 0.08), runSd=0.01, noise=0.02, seed=0, models=PARTB_MODELS):
    """Part b rows (levels '<model>|n32sq4', '|n32sq4liar', '|n32', '|n32liar', seeds 900-909) with known effects: every adapter
    gains gain4 over the warm start in squads of 4 and gain1 on one board, plus a run effect shared by both arms (so each run's
    difference is exactly gain4 - gain1); honest loss with liars loss[0] in squads of 4 and loss[1] on one board."""
    rng, rows = np.random.default_rng(seed), []
    e = {s: rng.normal(0, noise) for s in BIG_SEEDS}
    for m in models:
        u = 0.0 if m == 'warm' else rng.normal(0, runSd)
        for (c, l), g, ls, base in ((PARTB_TESTS['squads4'], gain4, loss[0], 0.32), (PARTB_TESTS['oneBoard'], gain1, loss[1], 0.30)):
            for s in BIG_SEEDS:
                w = base + e[s] + (0.0 if m == 'warm' else g + u)
                rows += [dict(level=f'{m}|{c}', seed=s, N=32, nLiar=0, welfare=w, welfareHonest=w),
                         dict(level=f'{m}|{l}', seed=s, N=32, nLiar=8, welfare=w - ls, welfareHonest=w - ls)]
    return rows

## Tests

In [ ]:
# test_c1_report's rows (AC-9): syntheticRows20 has every planned cell once (23 new with liars, 15 clean, 16 reused), the noise
# shared by every arm on a seed; syntheticPartB has the 4 part-b levels of 9 models on seeds 900-909
rows = syntheticRows20()
assert sorted((r['level'], r['seed']) for r in rows) == sorted(planned20()) and len(rows) == 54
by = {(r['level'], r['seed']): r['honestPooled'] for r in rows}
assert all(abs(by[('bridge1|independent', s)] - by[('colonyGuards|independent', s)] - 0.141) < 1e-12 for s in range(1800, 1808))
assert {r['nLiar'] for r in rows if r['level'].endswith('|clean')} == {0} and len(json.loads(rows[0]['agentTurns'])) == 64
assert len(syntheticPartB()) == 9 * 4 * 10 and syntheticRows20(seed=1) != rows

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()